# Corpus and runtime audit

Two existing MIT-licensed games, simulated playthroughs, no human participants. Read provenance before interpreting sample sizes. All cells below were executed.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.metrics import *


In [2]:
manifest = json.loads((ROOT/'data/source_manifest.json').read_text())
pd.DataFrame(manifest['files'])

,id,upstream_path,author,title,local_path,sha256,bytes
0,the_intercept,Stories/The Intercept/TheIntercept.ink,inkle,The Intercept,data/raw/the_intercept.ink,e89c3763d72d1a0ee4ea59e48881a723bac1c540fdae55...,100535
1,emoji,Stories/LD41 Emoji/LD41 Emoji.ink,Pat Scott,LD41 Emoji (alarm-clock title),data/raw/emoji.ink,2a93efe571e1ca8fc05ce30f228d911aa032eb1ad743cd...,15362


In [3]:
summary = pd.read_json(ROOT/'data/processed/extraction_summary.json')
summary[['game','runs','eligible_states','retained_states','extracted_edges','error_runs','truncated']]

,game,runs,eligible_states,retained_states,extracted_edges,error_runs,truncated
0,the_intercept,200,3403,114,349,64,0
1,emoji,200,4144,120,330,0,5


## One observed decision
The context and responses are creator-authored source material. Outcome IDs encode an observable proxy; they are not independent human semantic annotations.

In [4]:
contexts = json.loads((ROOT/'data/processed/the_intercept.contexts.json').read_text())
c = contexts[0]
print(c['context'])
pd.DataFrame(c['edges'])[['action_text','text','delta','terminal']]

"I saw Hooper take it."
"I see." He is starting to lose his patience. I have seen Harris angry a few times, with lackeys and secretaries. But never with us. With the 'brains' he has always been cautious, treating us like children.
And now I see that, like a father, he wants to smack us when we disobey him.
"Just get to the truth, man. Every <i>minute</i> matters."


,action_text,text,delta,terminal
0,Tell the truth,"""Very well. I see there's no point in covering...",{},False
1,Persist with this,"""This is the truth.""\nI have become, somehow, ...",{},False


In [5]:
import subprocess
r = subprocess.run(['node','scripts/verify_replay.cjs'],cwd=ROOT,text=True,capture_output=True,check=True)
print(r.stdout)

679 transitions replayed exactly.



### Interpretation
A valid local context can precede an error later in the full playthrough. Invalid immediate successors are excluded and logged. These states do not constitute independent games or writers.